In [1]:
# Cell 0 — Path setup
import sys, os
from pathlib import Path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name in ("notebooks", "current notebooks") or not (REPO_ROOT / "scripts").is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT = {REPO_ROOT}")

REPO_ROOT = /Users/JanRovirosaIlla/DeepMarkovResearch


In [2]:
# Cell 1 — Imports
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from sklearn.feature_selection import mutual_info_classif

from scripts.config import make_config
from scripts.data import load_master_dataset, preprocess_features, build_all_splits
from scripts.bins import compute_X_t, build_all_configs
from scripts.models import StateConditionedNet
from scripts.train import (
    train_one_run, build_loaders, MasterDataset,
    cache_model, load_cached_model, is_cached,
    create_soft_labels_batch,
)
from scripts.eval import evaluate_model, evaluate_baselines

print("Imports OK")

Imports OK


In [3]:
# Cell 2 — Experiment configuration
cfg = make_config()   # hyperparameters only

H        = 10
N_BINS   = 55
N_FEAT   = 30
SWEEP_SEED = 0   # single seed for all sweep sections

RESULTS_DIR = REPO_ROOT / "results" / "signal_search"
CACHE_DIR   = RESULTS_DIR / "cache"
FIG_DIR     = RESULTS_DIR / "figures"
for d in [RESULTS_DIR, CACHE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")
print(f"RESULTS_DIR = {RESULTS_DIR}")

DEVICE = cpu
RESULTS_DIR = /Users/JanRovirosaIlla/DeepMarkovResearch/results/signal_search


In [4]:
# Cell 3 — Data loading (shared by all sections)
prices, F_raw, feature_cols = load_master_dataset("dataset")
splits   = build_all_splits(prices, [H])
F_normed = preprocess_features(F_raw, splits[H]["idx_train"])

train_end = splits[H]["idx_train"][-1] + 1
X_t_all, N_XT, edges_xt = compute_X_t(prices, cfg.n_xt_target, train_end)

configs = build_all_configs(
    prices, F_normed, X_t_all,
    horizons=[H], n_bins_list=[N_BINS],
    N_XT=N_XT, edges_xt=edges_xt, splits=splits,
    sigma_anchor=cfg.sigma_anchor,
)
c = configs[(H, N_BINS)]
N_actual  = c["N_actual"]
idx_train = c["idx_train"]
idx_val   = c["idx_val"]
idx_test  = c["idx_test"]

print(f"Splits: train={len(idx_train)}, val={len(idx_val)}, test={len(idx_test)}")
print(f"N_actual={N_actual}, N_XT={N_XT}, c['sigma']={c['sigma']:.4f}")

Splits: train=1650, val=354, test=354
N_actual=55, N_XT=55, c['sigma']=1.0000


In [5]:
# Cell 4 — MI feature ranking (shared by Sections 3, 5)
print("Computing MI scores on aligned training data...")
mi_scores = mutual_info_classif(
    F_normed[idx_train], c["y_all"][idx_train],
    random_state=0, n_neighbors=5,
)
ranking = np.argsort(mi_scores)[::-1]  # descending MI
sel_idx = ranking[:N_FEAT]
F_sub   = F_normed[:, sel_idx]         # (T_all, 30)

print(f"Top-{N_FEAT} MI feature indices: {sel_idx.tolist()}")
print(f"Top-5 MI scores: {mi_scores[sel_idx[:5]].round(5).tolist()}")

Computing MI scores on aligned training data...


Top-30 MI feature indices: [94, 84, 23, 123, 177, 54, 147, 145, 143, 16, 184, 185, 95, 179, 97, 188, 31, 52, 116, 47, 30, 39, 131, 101, 176, 96, 191, 60, 104, 76]
Top-5 MI scores: [0.17334, 0.16985, 0.16866, 0.16743, 0.16662]


---
## Section 1 — Consistency Diagnostics

Verify that the pipeline uses the corrected forward-looking label definition
and document the full training configuration.

In [6]:
# Section 1 — Consistency Diagnostics
from scripts.bins import compute_sigma, get_edges
from scripts.data import compute_returns

# 1) Label definition check
# bins.py: R_h = compute_returns(prices[1:], h)
# → Y_t^(h) = bin((prices[t+1+h] − prices[t+1]) / prices[t+1])
# This is strictly forward-looking; h=1 is NOT identical to X_t.
label_def = "Y_t^h = bin((prices[t+1+h] - prices[t+1]) / prices[t+1])"

# Leakage check: at h=1, Y_t should NOT equal X_t
R_1 = compute_returns(prices[1:], 1)
R_xt = compute_returns(prices, 1)
from scripts.bins import assign_bins, get_edges
R_train_xt = R_xt[:train_end]
N_xt_check, edges_xt_check = get_edges(R_train_xt, 55)
X_t_check = assign_bins(R_xt, edges_xt_check)
R1_train = R_1[:train_end]
_, edges_y1 = get_edges(R1_train, 55)
Y_t1 = assign_bins(R_1, edges_y1)
T_min = min(len(X_t_check), len(Y_t1))
frac_equal = float((X_t_check[:T_min] == Y_t1[:T_min]).mean())
assert frac_equal < 0.99, f"Leakage! h=1 labels identical to X_t: frac={frac_equal:.4f}"
label_check = f"PASS (h=1 fraction equal to X_t: {frac_equal:.4f}, must be <0.99)"

# 2) Sigma (label smoothing)
sigma_anchor_val = cfg.sigma_anchor
# For N=55 with single-element n_bins_list, sigma = sigma_anchor * 1.0
sigma_actual = c["sigma"]
loss_type = "soft Gaussian KL (sigma > 0)" if sigma_actual > 0 else "hard cross-entropy (sigma = 0)"

# 3) Model input description
model_input = "StateConditionedNet: cat([one_hot(X_t, N_XT), F_t], dim=-1) -> MLP -> logits"

# 4) Cache integrity: list OptimalSetup caches
opt_cache = REPO_ROOT / "results" / "optimal_setup" / "cache"
opt_caches = sorted(opt_cache.glob("*.pt")) if opt_cache.exists() else []
cache_status = f"{len(opt_caches)} OptimalSetup cache files found under {opt_cache}"

diag = {
    "horizon": H,
    "N_bins": N_BINS,
    "N_actual": int(N_actual),
    "N_XT": int(N_XT),
    "n_features_base": int(N_FEAT),
    "label_definition": label_def,
    "label_check_h1": label_check,
    "sigma_anchor": float(sigma_anchor_val),
    "sigma_actual_N55": float(sigma_actual),
    "training_loss": loss_type,
    "model_input": model_input,
    "architecture": list(cfg.hidden_dims),
    "dropout": float(cfg.dropout),
    "lr": float(cfg.lr),
    "weight_decay": float(cfg.weight_decay),
    "max_epochs": int(cfg.max_epochs),
    "patience": int(cfg.patience),
    "train_size": int(len(idx_train)),
    "val_size": int(len(idx_val)),
    "test_size": int(len(idx_test)),
    "optimal_setup_cache_status": cache_status,
}

# Save JSON
with open(RESULTS_DIR / "diagnostic_config.json", "w") as f:
    json.dump(diag, f, indent=2)

# Pretty print
print("=" * 60)
print("DIAGNOSTIC REPORT — h=10, N=55")
print("=" * 60)
for k, v in diag.items():
    print(f"  {k:<35}: {v}")
print("=" * 60)
print(f"Saved: {RESULTS_DIR / 'diagnostic_config.json'}")

DIAGNOSTIC REPORT — h=10, N=55
  horizon                            : 10
  N_bins                             : 55
  N_actual                           : 55
  N_XT                               : 55
  n_features_base                    : 30
  label_definition                   : Y_t^h = bin((prices[t+1+h] - prices[t+1]) / prices[t+1])
  label_check_h1                     : PASS (h=1 fraction equal to X_t: 0.0199, must be <0.99)
  sigma_anchor                       : 1.0
  sigma_actual_N55                   : 1.0
  training_loss                      : soft Gaussian KL (sigma > 0)
  model_input                        : StateConditionedNet: cat([one_hot(X_t, N_XT), F_t], dim=-1) -> MLP -> logits
  architecture                       : [64, 128, 256, 128, 64]
  dropout                            : 0.2
  lr                                 : 0.001
  weight_decay                       : 0.0001
  max_epochs                         : 100
  patience                           : 10
  train_size    

---
## Section 2 — Baseline Sanity Check

Recompute all baselines for (h=10, N=55) and verify uniform NLL = log(N_bins).

In [7]:
# Section 2 — Baseline Sanity Check
baselines = evaluate_baselines(c, X_t_all, N_XT, cfg.alpha_grid, cfg.tau_grid)
uniform_nll = float(np.log(N_actual))

# Verify uniform NLL = log(N)
expected_uniform = float(np.log(N_bins := N_BINS))
assert abs(uniform_nll - expected_uniform) < 0.01, (
    f"N_actual={N_actual} != N_BINS={N_BINS}: uniform NLL mismatch"
)

baseline_rows = [
    {"baseline": "uniform",  "test_nll": uniform_nll,
     "note": f"= log({N_actual}) = {uniform_nll:.6f}"},
    {"baseline": "marginal", "test_nll": -baselines["marginal"]["test_ll"],
     "note": ""},
    {"baseline": "additive", "test_nll": -baselines["additive"]["test_ll"],
     "note": f"best alpha={baselines['additive']['alpha']}"},
    {"baseline": "backoff",  "test_nll": -baselines["backoff"]["test_ll"],
     "note": f"best alpha={baselines['backoff']['alpha']}, tau={baselines['backoff']['tau']}"},
]
df_bl = pd.DataFrame(baseline_rows)
df_bl.to_csv(RESULTS_DIR / "baseline_check.csv", index=False)

print("=" * 55)
print("BASELINE SANITY CHECK — h=10, N=55")
print("=" * 55)
print(df_bl.to_string(index=False))
print("=" * 55)
print(f"\nUniform NLL = log({N_actual}) = {uniform_nll:.6f}  ✓" if abs(uniform_nll - expected_uniform) < 0.01 else "MISMATCH")

BASELINE SANITY CHECK — h=10, N=55
baseline  test_nll                      note
 uniform  4.007333      = log(55) = 4.007333
marginal  4.007333                          
additive  4.007645           best alpha=10.0
 backoff  4.007273 best alpha=10.0, tau=1000

Uniform NLL = log(55) = 4.007333  ✓


---
## Section 3 — Sigma (Label Smoothing) Sweep

Train `StateConditionedNet` (n_feat=30, h=10, N=55) with varying sigma_anchor.

**Note on sigma=0.0:** `create_soft_labels_batch` divides by σ² so σ=0 is undefined.
We substitute σ=1e-9, which gives a numerically one-hot distribution (equivalent to
hard cross-entropy in float32 precision). This is noted in the results.

For N=55 with `n_bins_list=[55]`, the anchor edges equal the target edges, so
`c["sigma"] = sigma_anchor × 1.0`. Sigma values are therefore passed directly
to `train_one_run` without rebuilding configs.

In [8]:
# Section 3 — Sigma Sweep
SIGMA_ANCHORS = [0.0, 0.1, 0.25, 0.5, 1.0]
# sigma=0.0 → use 1e-9 (numerically hard CE; see note above)
SIGMA_TRAIN   = [1e-9 if s == 0.0 else s for s in SIGMA_ANCHORS]

def _sigma_str(s):
    """Safe filename string for a sigma value."""
    return f"{s:.4f}".replace(".", "p")

def _set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

tr_loader_30, val_loader_30, te_loader_30 = build_loaders(
    c, F_sub, X_t_all,
    batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
)
tr_eval_30 = DataLoader(
    MasterDataset(F_sub, X_t_all, c["y_all"], idx_train),
    batch_size=cfg.batch_eval, shuffle=False,
)

sigma_rows = []
for sigma_anchor, sigma_train in zip(SIGMA_ANCHORS, SIGMA_TRAIN):
    sstr = _sigma_str(sigma_anchor)
    cache_path = CACHE_DIR / f"sigswp_sa{sstr}_h{H}_N{N_BINS}_nf{N_FEAT}.pt"

    _set_seed(SWEEP_SEED)
    m = StateConditionedNet(
        n_feat=N_FEAT, n_xt_states=N_XT, n_output=N_actual,
        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout,
    )

    if is_cached(cache_path):
        print(f"  sigma_anchor={sigma_anchor}: loading cache...")
        load_cached_model(m, cache_path)
    else:
        print(f"  sigma_anchor={sigma_anchor} (sigma_train={sigma_train:.2e}): training...")
        best_state, _ = train_one_run(
            m, tr_loader_30, val_loader_30, N_actual, sigma_train, DEVICE,
            lr=cfg.lr, weight_decay=cfg.weight_decay,
            max_epochs=cfg.max_epochs, patience=cfg.patience,
            grad_clip=cfg.grad_clip,
        )
        cache_model(best_state, cache_path)

    m.to(DEVICE); m.eval()
    res_tr  = evaluate_model(m, tr_eval_30,   N_actual, DEVICE)
    res_val = evaluate_model(m, val_loader_30, N_actual, DEVICE)
    res_te  = evaluate_model(m, te_loader_30,  N_actual, DEVICE)

    row = {
        "sigma_anchor": sigma_anchor,
        "sigma_train":  sigma_train,
        "train_nll":    -res_tr["mean_ll"],
        "val_nll":      -res_val["mean_ll"],
        "test_nll":     -res_te["mean_ll"],
        "gen_gap":      (-res_te["mean_ll"]) - (-res_tr["mean_ll"]),
    }
    sigma_rows.append(row)
    print(f"    train={row['train_nll']:.4f} val={row['val_nll']:.4f} "
          f"test={row['test_nll']:.4f} gap={row['gen_gap']:.4f}")

df_sigma = pd.DataFrame(sigma_rows)
df_sigma.to_csv(RESULTS_DIR / "sigma_sweep.csv", index=False)

print("\n", df_sigma[["sigma_anchor", "train_nll", "val_nll", "test_nll", "gen_gap"]].to_string(index=False))

# Best sigma = lowest val_nll
best_idx     = df_sigma["val_nll"].idxmin()
best_sigma   = float(df_sigma.loc[best_idx, "sigma_train"])
best_sa      = float(df_sigma.loc[best_idx, "sigma_anchor"])
best_val_nll = float(df_sigma.loc[best_idx, "val_nll"])
print(f"\nBest sigma_anchor = {best_sa} (val_nll = {best_val_nll:.4f})")

  sigma_anchor=0.0 (sigma_train=1.00e-09): training...


    train=4.0052 val=4.0142 test=4.0091 gap=0.0039
  sigma_anchor=0.1 (sigma_train=1.00e-01): training...


    train=4.0052 val=4.0142 test=4.0091 gap=0.0039
  sigma_anchor=0.25 (sigma_train=2.50e-01): training...


    train=4.0052 val=4.0142 test=4.0091 gap=0.0039
  sigma_anchor=0.5 (sigma_train=5.00e-01): training...


    train=4.0037 val=4.0140 test=4.0084 gap=0.0046
  sigma_anchor=1.0 (sigma_train=1.00e+00): training...


    train=3.8835 val=3.9942 test=4.0060 gap=0.1225

  sigma_anchor  train_nll  val_nll  test_nll  gen_gap
         0.00   4.005169 4.014247  4.009116 0.003947
         0.10   4.005169 4.014247  4.009116 0.003947
         0.25   4.005167 4.014247  4.009112 0.003945
         0.50   4.003734 4.013999  4.008367 0.004632
         1.00   3.883516 3.994162  4.006007 0.122491

Best sigma_anchor = 1.0 (val_nll = 3.9942)


---
## Section 4 — Optimization Budget Test

Test whether early stopping is cutting off convergence.

- **Budget A**: original (max_epochs=100, patience=10)
- **Budget B**: extended (max_epochs=200, patience=25)

Both use the best sigma from Section 3.

In [9]:
# Section 4 — Training Budget Test
budget_configs = [
    {"label": "A_orig",  "max_epochs": cfg.max_epochs, "patience": cfg.patience},
    {"label": "B_ext",   "max_epochs": 200,             "patience": 25},
]

budget_rows = []
for bc in budget_configs:
    sstr = _sigma_str(best_sa)
    cache_path = CACHE_DIR / f"budget_{bc['label']}_sa{sstr}_h{H}_N{N_BINS}_nf{N_FEAT}.pt"

    _set_seed(SWEEP_SEED)
    m_b = StateConditionedNet(
        n_feat=N_FEAT, n_xt_states=N_XT, n_output=N_actual,
        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout,
    )

    if is_cached(cache_path):
        print(f"  Budget {bc['label']}: loading cache...")
        load_cached_model(m_b, cache_path)
    else:
        print(f"  Budget {bc['label']}: max_epochs={bc['max_epochs']}, patience={bc['patience']}, training...")
        best_state, hist = train_one_run(
            m_b, tr_loader_30, val_loader_30, N_actual, best_sigma, DEVICE,
            lr=cfg.lr, weight_decay=cfg.weight_decay,
            max_epochs=bc["max_epochs"], patience=bc["patience"],
            grad_clip=cfg.grad_clip,
        )
        cache_model(best_state, cache_path)
        print(f"    stopped at epoch {len(hist['train_loss'])}")

    m_b.to(DEVICE); m_b.eval()
    res_tr  = evaluate_model(m_b, tr_eval_30,   N_actual, DEVICE)
    res_val = evaluate_model(m_b, val_loader_30, N_actual, DEVICE)
    res_te  = evaluate_model(m_b, te_loader_30,  N_actual, DEVICE)

    row = {
        "budget":       bc["label"],
        "max_epochs":   bc["max_epochs"],
        "patience":     bc["patience"],
        "sigma_anchor": best_sa,
        "train_nll":    -res_tr["mean_ll"],
        "val_nll":      -res_val["mean_ll"],
        "test_nll":     -res_te["mean_ll"],
    }
    budget_rows.append(row)
    print(f"    train={row['train_nll']:.4f} val={row['val_nll']:.4f} test={row['test_nll']:.4f}")

df_budget = pd.DataFrame(budget_rows)
df_budget.to_csv(RESULTS_DIR / "training_budget.csv", index=False)

print("\n", df_budget[["budget", "max_epochs", "patience", "train_nll", "val_nll", "test_nll"]].to_string(index=False))

nll_A = float(df_budget[df_budget["budget"] == "A_orig"]["test_nll"])
nll_B = float(df_budget[df_budget["budget"] == "B_ext"]["test_nll"])
print(f"\nExtended budget improvement: ΔNLL = {nll_A - nll_B:+.4f} nats")

  Budget A_orig: max_epochs=100, patience=10, training...


    stopped at epoch 30
    train=3.8835 val=3.9942 test=4.0060
  Budget B_ext: max_epochs=200, patience=25, training...


    stopped at epoch 45
    train=3.8835 val=3.9942 test=4.0060

 budget  max_epochs  patience  train_nll  val_nll  test_nll
A_orig         100        10   3.883516 3.994162  4.006007
 B_ext         200        25   3.883516 3.994162  4.006007

Extended budget improvement: ΔNLL = +0.0000 nats


/var/folders/xp/8yb64l_j1wd31jcqx1l7l_wr0000gp/T/ipykernel_3490/706474663.py:54: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  nll_A = float(df_budget[df_budget["budget"] == "A_orig"]["test_nll"])
/var/folders/xp/8yb64l_j1wd31jcqx1l7l_wr0000gp/T/ipykernel_3490/706474663.py:55: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  nll_B = float(df_budget[df_budget["budget"] == "B_ext"]["test_nll"])


---
## Section 5 — Feature Dimension Sweep (h=10)

Using the best sigma from Section 3, evaluate n_features ∈ {30, 50, 194}.
Feature ranking uses MI computed on training data only.

In [10]:
# Section 5 — Feature Dimension Sweep
FEAT_SIZES = [30, 50, 194]

feat_rows = []
for n_feat in FEAT_SIZES:
    sel_k = ranking[:n_feat] if n_feat < 194 else np.arange(194)
    F_k   = F_normed[:, sel_k]
    sstr  = _sigma_str(best_sa)
    cache_path = CACHE_DIR / f"featswp_nf{n_feat}_sa{sstr}_h{H}_N{N_BINS}.pt"

    _set_seed(SWEEP_SEED)
    m_f = StateConditionedNet(
        n_feat=n_feat, n_xt_states=N_XT, n_output=N_actual,
        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout,
    )
    tr_f, val_f, te_f = build_loaders(
        c, F_k, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    if is_cached(cache_path):
        print(f"  n_feat={n_feat}: loading cache...")
        load_cached_model(m_f, cache_path)
    else:
        print(f"  n_feat={n_feat}: training with best_sigma={best_sa}...")
        best_state, _ = train_one_run(
            m_f, tr_f, val_f, N_actual, best_sigma, DEVICE,
            lr=cfg.lr, weight_decay=cfg.weight_decay,
            max_epochs=cfg.max_epochs, patience=cfg.patience,
            grad_clip=cfg.grad_clip,
        )
        cache_model(best_state, cache_path)

    m_f.to(DEVICE); m_f.eval()
    tr_eval_f = DataLoader(
        MasterDataset(F_k, X_t_all, c["y_all"], idx_train),
        batch_size=cfg.batch_eval, shuffle=False,
    )
    r_tr  = evaluate_model(m_f, tr_eval_f, N_actual, DEVICE)
    r_val = evaluate_model(m_f, val_f,     N_actual, DEVICE)
    r_te  = evaluate_model(m_f, te_f,      N_actual, DEVICE)

    row = {
        "n_features": n_feat,
        "sigma_anchor": best_sa,
        "train_nll":  -r_tr["mean_ll"],
        "val_nll":    -r_val["mean_ll"],
        "test_nll":   -r_te["mean_ll"],
        "gen_gap":    (-r_te["mean_ll"]) - (-r_tr["mean_ll"]),
    }
    feat_rows.append(row)
    print(f"    train={row['train_nll']:.4f} val={row['val_nll']:.4f} "
          f"test={row['test_nll']:.4f} gap={row['gen_gap']:.4f}")

df_feat = pd.DataFrame(feat_rows)
df_feat.to_csv(RESULTS_DIR / "feature_sweep.csv", index=False)

print("\n", df_feat.to_string(index=False))

  n_feat=30: training with best_sigma=1.0...


    train=3.8835 val=3.9942 test=4.0060 gap=0.1225
  n_feat=50: training with best_sigma=1.0...


    train=3.8889 val=3.9896 test=4.0523 gap=0.1634
  n_feat=194: training with best_sigma=1.0...


    train=3.9297 val=3.9979 test=4.0078 gap=0.0781

  n_features  sigma_anchor  train_nll  val_nll  test_nll  gen_gap
         30           1.0   3.883516 3.994162  4.006007 0.122491
         50           1.0   3.888862 3.989562  4.052288 0.163426
        194           1.0   3.929670 3.997911  4.007802 0.078132


---
## Section 6 — Final Comparison

Collect best neural result across all sweeps and compare to baselines.

In [11]:
# Section 6 — Final Comparison
uniform_nll  = float(np.log(N_actual))
marginal_nll = float(-baselines["marginal"]["test_ll"])
additive_nll = float(-baselines["additive"]["test_ll"])

# Collect all neural test NLLs from sweeps
all_neural_nlls = (
    list(df_sigma["test_nll"]) +
    list(df_budget["test_nll"]) +
    list(df_feat["test_nll"])
)
best_neural_nll = float(min(all_neural_nlls))

# Source of best
from_sigma  = df_sigma["test_nll"].min()
from_budget = df_budget["test_nll"].min()
from_feat   = df_feat["test_nll"].min()
best_source = (
    "sigma_sweep" if from_sigma == best_neural_nll else
    "budget_test" if from_budget == best_neural_nll else
    "feature_sweep"
)

delta_vs_uniform  = uniform_nll  - best_neural_nll   # positive = neural beats uniform
delta_vs_marginal = marginal_nll - best_neural_nll
delta_vs_additive = additive_nll - best_neural_nll

THRESHOLD = 0.001   # nats: target improvement

summary = pd.DataFrame([
    {"metric": "uniform_nll",        "value": uniform_nll},
    {"metric": "marginal_nll",       "value": marginal_nll},
    {"metric": "additive_nll",       "value": additive_nll},
    {"metric": "best_neural_nll",    "value": best_neural_nll},
    {"metric": "best_source",        "value": best_source},
    {"metric": "delta_vs_uniform",   "value": delta_vs_uniform},
    {"metric": "delta_vs_marginal",  "value": delta_vs_marginal},
    {"metric": "delta_vs_additive",  "value": delta_vs_additive},
    {"metric": "threshold",          "value": THRESHOLD},
    {"metric": "target_met",         "value": str(delta_vs_uniform >= THRESHOLD)},
])
summary.to_csv(RESULTS_DIR / "final_summary.csv", index=False)

print("=" * 60)
print("SIGNAL SEARCH — FINAL SUMMARY (h=10, N=55)")
print("=" * 60)
print(summary.to_string(index=False))
print("=" * 60)

if delta_vs_uniform >= THRESHOLD:
    print(f"\n✓ TARGET MET: best neural NLL beats uniform by {delta_vs_uniform:.4f} nats "
          f"(≥ {THRESHOLD} threshold). Source: {best_source}")
else:
    print(f"\n✗ TARGET NOT MET: best neural NLL is {-delta_vs_uniform:.4f} nats ABOVE uniform.")
    print(f"  Best neural NLL = {best_neural_nll:.4f}, uniform = {uniform_nll:.4f}")
    print(f"  Source of best: {best_source}")
    print()
    print("  Sigma sweep results:")
    print(df_sigma[["sigma_anchor", "val_nll", "test_nll"]].to_string(index=False))
    print()
    print("  Budget test results:")
    print(df_budget[["budget", "patience", "val_nll", "test_nll"]].to_string(index=False))
    print()
    print("  Feature sweep results:")
    print(df_feat[["n_features", "val_nll", "test_nll", "gen_gap"]].to_string(index=False))

SIGNAL SEARCH — FINAL SUMMARY (h=10, N=55)
           metric       value
      uniform_nll    4.007333
     marginal_nll    4.007333
     additive_nll    4.007645
  best_neural_nll    4.006007
      best_source sigma_sweep
 delta_vs_uniform    0.001326
delta_vs_marginal    0.001325
delta_vs_additive    0.001638
        threshold       0.001
       target_met        True

✓ TARGET MET: best neural NLL beats uniform by 0.0013 nats (≥ 0.001 threshold). Source: sigma_sweep
